# STRATA dentro do llama.cpp: tokens por segundo e perplexidade, contra Q5_K_M e Q6_K

**Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente de execução*. Escolha **TPU v6e-1** com **RAM alta** (o que interessa é a CPU da máquina, que costuma ter muitos núcleos). Se a primeira célula disser que a máquina não serve, troque para **A100** com **RAM alta**. Depois *Executar tudo*.

O notebook:
1. compila o llama.cpp (commit fixo) com os tipos novos do STRATA (`kernel/ggml`): os níveis 1 e 2 do modo residual em blocos montados na carga;
2. gera os GGUF do Llama-3.2-1B: BF16, Q4_K_M, Q5_K_M, Q6_K e os níveis 1 e 2 do STRATA (feitos a partir do Q4_K_M + pesos originais, como o leitor faz ao carregar um `.strata` 0.3);
3. confere que o kernel dá o resultado certo nesta CPU;
4. mede a perplexidade no wikitext-2 de teste e a divergência KL contra o BF16 com o próprio `llama-perplexity` (o kernel de verdade, não emulação);
5. mede tokens por segundo com o `llama-bench`, alternando os modelos em rodadas, com 1 thread, com os núcleos físicos e com todos.

O tempo depende de quantos núcleos a máquina tiver; a perplexidade do wikitext inteiro é a parte mais demorada (para encurtar, mude `CHUNKS` na célula dela).

In [ ]:
import os, platform, subprocess
info = open('/proc/cpuinfo').read()
modelo = next((l.split(':', 1)[1].strip() for l in info.splitlines() if l.startswith('model name')), '?')
N = os.cpu_count()
lscpu = subprocess.run(['lscpu'], capture_output=True, text=True).stdout
def campo(nome):
    return next((l.split(':', 1)[1].strip() for l in lscpu.splitlines() if l.startswith(nome)), '1')
FISICOS = int(campo('Core(s) per socket')) * int(campo('Socket(s)'))
RAM = int(next(l for l in open('/proc/meminfo') if l.startswith('MemTotal')).split()[1]) / 1e6
AVX512 = ' avx512f' in info
print(f'CPU: {modelo} | {platform.machine()} | {N} threads lógicas, {FISICOS} núcleos físicos | RAM {RAM:.0f} GB | AVX2: {" avx2" in info} | AVX-512: {AVX512}')
assert platform.machine() == 'x86_64' and ' avx2' in info, 'Esta máquina não serve (precisa de x86 com AVX2): troque o tipo de ambiente de execução para A100 com RAM alta.'
if N < 8:
    print('Aviso: poucos núcleos. A medição funciona, mas uma máquina com mais núcleos (TPU v6e-1 ou A100) fica mais perto de um PC.')

In [ ]:
BRANCH = "claude/project-thread-qjksmf"
LLAMA_SHA = "2a667032dc788c3a071d538e0b572e45a3f12731"   # o commit em que o patch dos tipos foi feito
!rm -rf strata && git clone -q --depth 1 -b $BRANCH https://github.com/Cantyll/strata.git
!rm -rf llama.cpp && git init -q llama.cpp && cd llama.cpp && git remote add origin https://github.com/ggml-org/llama.cpp.git && git fetch -q --depth 1 origin $LLAMA_SHA && git checkout -q FETCH_HEAD
!sh strata/kernel/ggml/aplicar.sh llama.cpp
try:
    import torch
except ImportError:
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!pip install -q -U transformers datasets safetensors
!pip install -q -e llama.cpp/gguf-py
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DGGML_CUDA=OFF -DGGML_NATIVE=ON > /dev/null && cmake --build build -j {N} --target llama-quantize llama-bench llama-perplexity 2>&1 | grep -E ' error' ; ls llama.cpp/build/bin

Os GGUF. Os k-quants saem do `llama-quantize` padrão, sem imatrix; os níveis do STRATA saem do Q4_K_M + pesos BF16.

In [ ]:
from huggingface_hub import snapshot_download
src = snapshot_download("unsloth/Llama-3.2-1B", allow_patterns=["*.json", "*.safetensors", "tokenizer*"])
!mkdir -p /content/gguf && python llama.cpp/convert_hf_to_gguf.py {src} --outtype bf16 --outfile /content/gguf/BF16.gguf 2>&1 | tail -1
for q in ["Q4_K_M", "Q5_K_M", "Q6_K"]:
    !llama.cpp/build/bin/llama-quantize /content/gguf/BF16.gguf /content/gguf/{q}.gguf {q} > /content/gguf/{q}.log 2>&1
%cd /content/strata
for k in [1, 2]:
    !python tests/kernel/strata_gguf_niveis.py --model {src} --gguf /content/gguf/Q4_K_M.gguf --nivel {k} --out /content/gguf/STRATA_N{k}.gguf 2>&1 | tail -1
%cd /content
!ls -la /content/gguf

Conferência rápida do kernel nesta CPU (3 matrizes de cada tipo; tudo tem de sair OK).

In [ ]:
%cd /content/strata
!python tests/kernel/test_q4kr.py --model {src} --gguf /content/gguf/Q4_K_M.gguf --levels 1,2 --limit 3 --dot-rows 4 2>&1 | grep -E 'bits/peso|Error'
%cd /content

Perplexidade no wikitext-2 de teste (contexto 512), com o `llama-perplexity`. `CHUNKS = 0` usa o texto inteiro.

In [ ]:
import re, json
from datasets import load_dataset
CHUNKS = 0
open('/content/wiki.test.raw', 'w').write(''.join(load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')['text']))
MODELOS = ['BF16', 'Q4_K_M', 'Q5_K_M', 'STRATA_N1', 'Q6_K', 'STRATA_N2']
ppl = {}
for m in MODELOS:
    extra = f'--chunks {CHUNKS}' if CHUNKS else ''
    out = !llama.cpp/build/bin/llama-perplexity -m /content/gguf/{m}.gguf -f /content/wiki.test.raw -c 512 -t {N} {extra} 2>&1
    r = re.search(r'Final estimate: PPL = ([0-9.]+) \+/- ([0-9.]+)', '\n'.join(out))
    ppl[m] = (float(r.group(1)), float(r.group(2)))
    print(m, ppl[m], f'ΔPPL {100 * (ppl[m][0] / ppl["BF16"][0] - 1):+.2f}%')

Divergência KL contra o BF16 nos primeiros `KLD_CHUNKS` trechos: mede quanto a distribuição de probabilidade do próximo token se afasta da do modelo original. É bem menos ruidosa que a perplexidade (a perplexidade de um modelo quantizado pode sair até menor que a do original por acaso; a KL não). O arquivo de referência ocupa ~65 MB por trecho no disco.

In [ ]:
KLD_CHUNKS = 100
!llama.cpp/build/bin/llama-perplexity -m /content/gguf/BF16.gguf -f /content/wiki.test.raw -c 512 -t {N} --chunks {KLD_CHUNKS} --kl-divergence-base /content/kld_base.bin > /dev/null 2>&1
kld = {}
for m in MODELOS[1:]:
    out = !llama.cpp/build/bin/llama-perplexity -m /content/gguf/{m}.gguf --kl-divergence-base /content/kld_base.bin --kl-divergence -t {N} 2>&1
    txt = '\n'.join(out)
    k_ = re.search(r'Mean +KLD: +([0-9.]+) ± +([0-9.]+)', txt)
    top = re.search(r'Same top p: +([0-9.]+) ± +([0-9.]+) %', txt)
    kld[m] = (float(k_.group(1)), float(k_.group(2)), float(top.group(1)))
    print(m, kld[m])
!rm -f /content/kld_base.bin

Tokens por segundo com o `llama-bench`: geração de 128 tokens (`tg128`) com 1 thread, com os núcleos físicos e com todas as threads, e leitura de um prompt de 512 tokens (`pp512`) com os núcleos físicos. Os modelos são alternados em ordem aleatória a cada rodada, para que uma oscilação da máquina não favoreça nenhum.

In [ ]:
import random, statistics
BENCH = 'llama.cpp/build/bin/llama-bench'
RODADAS = 3
THREADS = sorted({1, FISICOS, N})
MODELOS_V = ['Q4_K_M', 'Q5_K_M', 'STRATA_N1', 'Q6_K', 'STRATA_N2']
vel = {}
def bench(m, t, p, n):
    out = subprocess.run([BENCH, '-m', f'/content/gguf/{m}.gguf', '-t', str(t), '-p', str(p), '-n', str(n), '-r', '3', '-o', 'json'],
                         capture_output=True, text=True).stdout
    return json.loads(out)[0]['avg_ts']
for r in range(RODADAS):
    ordem = MODELOS_V[:]
    random.shuffle(ordem)
    for m in ordem:
        for t in THREADS:
            vel.setdefault((m, f'tg128 t{t}'), []).append(bench(m, t, 0, 128))
        vel.setdefault((m, f'pp512 t{FISICOS}'), []).append(bench(m, FISICOS, 512, 0))
    print(f'rodada {r + 1} de {RODADAS} pronta')
TESTES = [f'tg128 t{t}' for t in THREADS] + [f'pp512 t{FISICOS}']
med = {k: statistics.median(v) for k, v in vel.items()}

In [ ]:
from IPython.display import Markdown, display
tam = {m: os.path.getsize(f'/content/gguf/{m}.gguf') / 2**30 for m in ['BF16'] + MODELOS_V}
L = [f'# STRATA no llama.cpp (Llama-3.2-1B)', '',
     f'CPU: {modelo}, {N} threads lógicas, {FISICOS} núcleos físicos, RAM {RAM:.0f} GB, AVX-512: {AVX512}. '
     f'Wikitext-2 de teste, contexto 512{f", {CHUNKS} trechos" if CHUNKS else ", inteiro"}; KL contra o BF16 nos {KLD_CHUNKS} primeiros trechos. '
     f'tok/s = mediana de {RODADAS} rodadas alternadas.', '',
     '| modelo | GGUF (GiB) | PPL | ΔPPL | KL (×1000) | mesmo 1º token | ' + ' | '.join(f'{t} (tok/s)' for t in TESTES) + ' |',
     '|---|---:|---:|---:|---:|---:|' + '---:|' * len(TESTES)]
for m in ['BF16'] + MODELOS_V:
    p, e = ppl[m]
    v = ' | '.join(f'{med[(m, t)]:.1f}' if (m, t) in med else '' for t in TESTES)
    kl = f'{1000 * kld[m][0]:.2f} ± {1000 * kld[m][1]:.2f} | {kld[m][2]:.2f}%' if m in kld else ' | '
    L.append(f'| {m} | {tam[m]:.3f} | {p:.4f} ± {e:.4f} | {100 * (p / ppl["BF16"][0] - 1):+.2f}% | {kl} | {v} |')
L += ['', 'Velocidade do STRATA em relação ao k-quant de mesma qualidade (1,00 = igual; abaixo de 1 = mais lento):', '',
      '| comparação | ' + ' | '.join(TESTES) + ' |', '|---|' + '---:|' * len(TESTES)]
for a, b in [('STRATA_N1', 'Q5_K_M'), ('STRATA_N2', 'Q6_K')]:
    L.append(f'| {a} / {b} | ' + ' | '.join(f'{med[(a, t)] / med[(b, t)]:.2f}' for t in TESTES) + ' |')
L += ['', 'Emulação anterior (Colab, seção 22 dos resultados): nível 1 +1,18%, nível 2 +0,31%, Q5_K_M +1,25%, Q6_K +0,49%.']
os.makedirs('/content/res', exist_ok=True)
open('/content/res/strata_llamacpp.md', 'w').write('\n'.join(L) + '\n')
json.dump({'cpu': modelo, 'threads': N, 'fisicos': FISICOS, 'ram_gb': RAM, 'avx512': AVX512, 'chunks': CHUNKS, 'ppl': ppl,
           'kld_chunks': KLD_CHUNKS, 'kld': kld,
           'tok_s': {f'{m}|{t}': v for (m, t), v in vel.items()}, 'gguf_gib': tam}, open('/content/res/strata_llamacpp.json', 'w'), indent=1)
display(Markdown('\n'.join(L)))

Cole a tabela na thread (ou baixe os arquivos abaixo).

In [ ]:
from google.colab import files
files.download('/content/res/strata_llamacpp.md')
files.download('/content/res/strata_llamacpp.json')